In [1]:
import pandas as pd
import ast
import numpy as np
import pyterrier as pt
from ir_measures import *
from pyterrier_t5 import MonoT5ReRanker

In [2]:
reranker = MonoT5ReRanker(model = "./mymodel-best-80/", batch_size = 32)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [3]:
def _sens_docs(qrels, run):
    if "sensitivity" in run.columns:
        run = run.drop(columns = ["sensitivity"])
    merged = pd.merge(run, ohsumed_docs, left_on = "doc_id", right_on = "docno")
    return merged.sensitivity.sum()
    
import ir_measures
sens_docs = ir_measures.define_byquery(
    _sens_docs, 
    name="sens_docs")

ohsumed_docs = pd.read_pickle("/nfs/primary/sas_reranker/ohsumed_docs_w_t5base_sensitivity.pkl")
queries = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/queries.pkl")
qrels = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/qrels.pkl")
bm25_results = pt.io.read_results("/nfs/primary/standard_run_files/ohsumed/bm25_pisa_1000.run")
valid_data = pd.merge(bm25_results, ohsumed_docs, on = "docno")

def evaluate():
    eval_results = pt.Experiment(
    [
        valid_data,
        pt.Transformer.from_df(valid_data) >> reranker
    ],
    queries,
    qrels,
    eval_metrics=[nDCG@10, sens_docs@10],
    names = [
        "BM25",
        "BM25 >> monoT5"
    ])
    print(eval_results)

    return {
        "nDCG@10" : eval_results["nDCG@10"].iloc[0],
        "sens_docs@10" : eval_results["nDCG@10"].iloc[0]
    }

evaluate()

monoT5: 100%|██████████| 3213/3213 [18:11<00:00,  2.94batches/s]


             name   nDCG@10  sens_docs@10
0            BM25  0.391683      1.339623
1  BM25 >> monoT5  0.498619      1.301887


{'nDCG@10': 0.39168253468820163, 'sens_docs@10': 0.39168253468820163}